# Load modules

In [22]:
import dask_jobqueue
import dask
import dask.distributed as dask_distributed
import os
import xarray as xr
import numpy as np
import pyproj

from gsw import f
import matplotlib.pyplot as plt
from scipy.interpolate import griddata
from scipy import stats
from scipy import signal

import functions

# Open dask-cluster

In [2]:
%%time
cluster = dask_jobqueue.SLURMCluster(cores=4,memory='20GB',processes=1,queue='base', walltime='20:00:00',interface='ib0',local_directory='$TMPDIR')
client = dask.distributed.Client(cluster)
cluster.scale(jobs=6)
client

/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/distributed/node.py:182: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 35875 instead
  f"Port {expected} is already in use.\n"


CPU times: user 530 ms, sys: 694 ms, total: 1.22 s
Wall time: 4.73 s


Connection method: Cluster object,Cluster type: dask_jobqueue.SLURMCluster
Dashboard: http://172.18.4.115:35875/status,
Dashboard: http://172.18.4.115:35875/status,Workers: 0
Total threads: 0,Total memory: 0 B
Comm: tcp://172.18.4.115:45857,Workers: 0
Dashboard: http://172.18.4.115:35875/status,Total threads: 0
Started: Just now,Total memory: 0 B


# Define directories

In [20]:
hcs = 300 # horizontal chunk size
path_data = '/gxfs_work/geomar/smomw355/model_data/ocean-only/INALT60.L120-KRS0020/nemo/'
path_mask = path_data + 'suppl/2_INALT60.L120-KRS0020_mesh_mask.nc'
path_data += 'output/2_INALT60.L120-KRS0020_'

path_current = os.getcwd()
paths_RMS = !ls {os.path.join(path_current, 'RMS_w_model/') +'*'+'.nc'}
ds_RMS_model=xr.open_mfdataset(paths_RMS)

# Define period and region of interest

In [23]:
lon_c=15.7
lat_c=-37.8

pos_west,pos_east,pos_south,pos_north  = lon_c-2,lon_c+2,lat_c-2,lat_c+2

# choose output frequency
frq = '1d'  
# '1d' = daily means (full water column, levels 1 - 120)
# '4h' = 4h means (levels 2 - 41)
# '5d' = a snapshot each fifth day (levels 2 - 41)

# note that there is data from 2010 - 2017, however the first two years are associated with a secondary spin-up and should not be analyzed.
yeara = 2012
yearb = 2017
yearvec = np.arange(yeara,yearb+1)

# Read data

In [8]:
ds_mask = xr.open_dataset(path_mask,chunks={'x':hcs,'y':hcs})
ds_mask_region = ds_mask.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

#SSH
paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_T.nc'}
for year in yearvec[1:]:
   b = !ls {path_data + frq + '_' + str(year) + '*' + '_grid_T.nc'}
   paths = np.hstack([paths,b])

ds = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'deptht':1,'time_counter':-1}).squeeze().rename({'deptht':'z'})#[['vovecrtz']] # in m/s
ds_ssh_region = ds.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

#Vertical velocity
paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_W.nc'}
for year in yearvec[1:]:
   b = !ls {path_data + frq + '_' + str(year) + '*' + '_grid_W.nc'}
   paths = np.hstack([paths,b])

ds = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthw':1,'time_counter':-1}).squeeze().rename({'depthw':'z'})#[['vovecrtz']] # in m/s
ds_w_region = ds.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

#Zonal velocity
paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_U.nc'}
for year in yearvec[1:]:
   b = !ls {path_data + frq + '_' + str(year) + '*' + '_grid_U.nc'}
   paths = np.hstack([paths,b])

ds = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthu':1,'time_counter':-1}).squeeze().rename({'depthu':'z'})
ds_u_region = ds.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

#Meridional velocity
paths = !ls {path_data + frq + '_' + str(yeara) + '*' + '_grid_V.nc'}
for year in yearvec[1:]:
   b = !ls {path_data + frq + '_' + str(year) + '*' + '_grid_V.nc'}
   paths = np.hstack([paths,b])

ds = xr.open_mfdataset(paths,chunks={'x':hcs,'y':hcs,'depthv':1,'time_counter':-1}).squeeze().rename({'depthv':'z'})
ds_v_region = ds.where((ds_mask.nav_lat>pos_south-1.2) & (ds_mask.nav_lat<pos_north+1.2) & (ds_mask.nav_lon>pos_west-1.2) & (ds_mask.nav_lon<pos_east+1.2), drop=True)

#Resize dataset to the first 1000 m
mask_1000 = ds_ssh_region.z <= 1000
ds_mask_region = ds_mask_region.where(mask_1000, drop=True)
ds_v_region=ds_v_region.where(mask_1000, drop=True)
ds_u_region=ds_u_region.where(mask_1000, drop=True)

mask_1000_w = ds_w_region.z <= 1050
ds_w_region = ds_w_region.where(mask_1000_w, drop=True)

/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1233: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  value = value[(slice(None),) * axis + (subkey,)]
/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1233: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': Tr

# Define a regular metric grid (needed for the eSQG comparison) for interpolation

In [9]:
dx=2 #regular spacing in km
lon_grid=ds_w_region.nav_lon
lat_grid=ds_w_region.nav_lat

#Define the grid
a = pyproj.Transformer.from_crs(4326,3395).transform(lon_grid,lat_grid)
y3 = a[0]
x3 = a[1]
x3min = np.nanmin(x3,1)
y3min = np.nanmin(y3,0)
Y3min,X3min = np.meshgrid(y3min, x3min)
x1=(x3-X3min)/1000
y1=(y3-Y3min)/1000
x_len = int(np.floor(np.amax(x1))+1)
y_len = int(np.floor(np.amax(y1))+1)
x_dim = np.linspace(0, x_len, int(x_len/dx), endpoint=False)
y_dim = np.linspace(0, y_len, int(y_len/dx), endpoint=False)
x2, y2 = np.meshgrid(x_dim, y_dim)

#Retrieve longitudes and latitudes back from the interpolation 
x2_proj = x2*1000 + x3min[0]
y2_proj = y2*1000 + y3min[0]
transformer_back = pyproj.Transformer.from_crs(3395, 4326, always_xy=True)
lat2, lon2 = transformer_back.transform(y2_proj,x2_proj)

#Make square
size_x=len(lon2[0,:])
size_y=len(lon2[:,0])

if size_y>size_x:
    delta_points=size_y-size_x
    lon2 = lon2[:size_x,:]
    lat2 = lat2[:size_x,:]
else:
    delta_points=size_x-size_y
    lon2 = lon2[:,:size_y]
    lat2 = lat2[:,:size_y]

lon2 = lon2[50:-50,50:-50]
lat2 = lat2[50:-50,50:-50]

# Loop over each month to compute N$_0^2$ and c

In [ ]:
path_save_fig = os.path.join(path_current, 'Figure_optimization') 
os.makedirs(path_save_fig, exist_ok=True)
month_label=['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
lfilter=40 
for m in range(1,13):
    #Select all the vertical profile of w variance for a given month
    ds_select = ds_RMS_model.where(ds_RMS_model.time_counter.dt.month.isin([m]), drop=True)

    #Compute the RMS profile anomaly  
    mean_prof=ds_select.RMS_model.mean(dim='time_counter')
    ds_anomaly=ds_select.RMS_model-mean_prof
        
    #Find mean MLD
    MLD_model_m=ds_ssh_region.sel(time_counter=ds_select.time_counter).sobowlin
    
    MLD_gridded = xr.apply_ufunc(interp_to_grid,
        MLD_model_m.chunk(dict(y=-1, x=-1, time_counter=40)), x1, y1, x2, y2,
        input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
        output_core_dims=[["new_y","new_x"]],
        exclude_dims=set(("y","x")),
        vectorize = True,
        dask="parallelized",
        output_dtypes=[MLD_model_m.dtype]).rename({"new_y": "y", "new_x": "x"})
    
    #Resize datasets to be square
    size_x=len(MLD_gridded.x)
    size_y=len(MLD_gridded.y)
    if size_y>size_x:
        MLD_final = MLD_gridded.isel(y=slice(0,size_x))
    else:
        MLD_final = MLD_gridded.isel(x=slice(0,size_y))
    
    MLD_final = MLD_final.isel(x=slice(50,-50),y=slice(50,-50)) #Resize to te region included in the analysis
    MLD_mean_month=MLD_final.mean(dim=['x','y']).mean().compute()

    #Compute the differences with the mean RMS profiles from below the mean MLD to 1,000 m
    mask_MLD = ds_ssh_region.z.where(mask_1000, drop=True) > MLD_mean_month
    ds_anomaly=ds_anomaly.where(mask_MLD, drop=True)
    RMSE_anomaly=((ds_anomaly**2).mean(dim='z'))**.5

    #Loop over the 10 days the closest to the mean RMS profile
    N2_list_month = np.zeros(10)
    cc_list_month = np.zeros(10)
    for day_sel, idd in zip(np.argsort(RMSE_anomaly.values)[:10], np.arange(10)):
        MLD_day=MLD_final.isel(time_counter=day_sel).mean(dim=['x','y']).compute()
        w_model = ds_w_region.sel(time_counter=ds_select.isel(time_counter=day_sel).time_counter)
        ssh_model = ds_ssh_region.sel(time_counter=ds_select.isel(time_counter=day_sel).time_counter).sossheig.where(ds_mask_region.tmask.isel(z=0)==1).squeeze()    
        u_model = ds_u_region.sel(time_counter=ds_select.isel(time_counter=day_sel).time_counter).vozocrtx.where(ds_mask_region.umask==1).squeeze()    
        v_model = ds_v_region.sel(time_counter=ds_select.isel(time_counter=day_sel).time_counter).vomecrty.where(ds_mask_region.umask==1).squeeze() 
        
        #Interpolate w on the vertical at SSH grid points since the eSQG will produce w estimates at SSH grid points
        w_org = w_model.vovecrtz  
        z1=w_model.z
        z2=ds_ssh_region.z.where(mask_1000, drop=True).rename({"z": "z_new"})
        
        w_gridded_vert = xr.apply_ufunc(interp_to_prof,
                             w_org.chunk(
                {"x": 100, "y": 100, "z":-1}), z1, z2,
                             input_core_dims=[["z"], ["z"], ["z_new"]],
                             output_core_dims=[["z_new"]],
                             exclude_dims=set(("z")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_org.dtype]).rename({"z_new": "z"})
    
        #Mask data
        w_gridded_vert=w_gridded_vert.where(ds_mask_region.tmask==1).squeeze()*60*60*24
    
        #Interpolate the fields on the horizontal on the regular 2-km grid
        ssh_gridded = xr.apply_ufunc(interp_to_grid,
                     ssh_model.chunk(
        {"x": -1, "y": -1}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[ssh_model.dtype]).rename({"new_y": "y", "new_x": "x"})
        
        w_gridded = xr.apply_ufunc(interp_to_grid,
                             w_gridded_vert.chunk(
                {"x": -1, "y": -1, "z":10}), x1, y1, x2, y2,
                             input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                             output_core_dims=[["new_y","new_x"]],
                             exclude_dims=set(("y","x")),
                             vectorize = True,
                             dask="parallelized",
                             output_dtypes=[w_gridded_vert.dtype]).rename({"new_y": "y", "new_x": "x"})
            
        u_gridded = xr.apply_ufunc(interp_to_grid,
                     u_model.chunk(
        {"x": -1, "y": -1, "z":10}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[u_model.dtype]).rename({"new_y": "y", "new_x": "x"})
    
        v_gridded = xr.apply_ufunc(interp_to_grid,
                     v_model.chunk(
        {"x": -1, "y": -1, "z":10}), x1, y1, x2, y2,
                     input_core_dims=[["y","x"], ["y","x"], ["y","x"], ["new_y","new_x"], ["new_y","new_x"]],
                     output_core_dims=[["new_y","new_x"]],
                     exclude_dims=set(("y","x")),
                     vectorize = True,
                     dask="parallelized",
                     output_dtypes=[v_model.dtype]).rename({"new_y": "y", "new_x": "x"})
    
        #Filter w at 40 km
        w_filtered = filtering_lanczos(w_gridded, lfilter).isel(x=slice(50,-50),y=slice(50,-50)).compute()
        ssh_filtered = filtering_lanczos(ssh_gridded, lfilter).compute()
        u_filtered = filtering_lanczos(u_gridded, lfilter)
        v_filtered = filtering_lanczos(v_gridded, lfilter)
        
        #Compute vorticity from u and v
        dudy = (u_filtered.shift(y=-1) - u_filtered.shift(y=1)) / (2*1000*dx)
        dvdx = (v_filtered.shift(x=-1) - v_filtered.shift(x=1)) / (2*1000*dx) 
        
        zeta_model=dvdx-dudy
        zeta_model = zeta_model.isel(x=slice(50,-50),y=slice(50,-50)).compute()
       
        # Optimize N_0^2 on the vertical profile of zeta variance (from below the MLD)
        mask_MLD = ds_ssh_region.z.where(mask_1000, drop=True) > MLD_mean_month
        RMS_model_zeta = (zeta_model**2).mean(dim=["x", "y"])**(1/2)
        RMS_model_interest=RMS_model_zeta.where(mask_MLD, drop=True)

        #Compute zeta from the model SSH applying the eSQG
        g = 9.81
        rho0=1.e+3
        f0=abs(np.mean(f(lat2)))
        gof0=g/f0
        z_model = -RMS_model_zeta.z.values

        #Prepare SSH field before applying the eSQG
        ssh_sqg = ssh_preprocessing_sqg(ssh_filtered)

        #Take the dimensions of the double periodic field
        M=len(ssh_sqg[:,0])
        L=len(ssh_sqg[0,:])
        kx,ky,kkx,kky,kk = get_kxky(L,M,2000,2000)
        
        N=len(z_model)

        N2_list=np.concatenate((np.arange(6e-7, 9e-7,1e-7),np.arange(1e-6, 9.1e-6,1e-6),np.arange(1e-5, 9.1e-5,1e-5)))
        delta_list=np.zeros(len(N2_list))+np.nan

        #Loop over the different N_0^2 values to try (in N2_list)
        for N02, n in zip(N2_list, np.arange(len(N2_list))):
            N0=np.sqrt(N02)        
            N0of0=N0/f0
            Rho0oN02og=rho0/N02/g
            goN02orho0=g/N02/rho0
            rho_cst=N0of0*rho0
            zz= z_model*N0of0
        
            zeta = sqg_rel_vort(ssh_sqg, M, L, N, kk, zz, gof0)[:,50:-50, 50:-50] #Reduce the field to remove spectral residuals
        
            #Compute RMS
            RMS_one_day=np.sqrt(np.nanmean((zeta**2),axis=(1,2)))
            RMS_sqg_interest=RMS_one_day[np.where(mask_MLD)[0]]

            #Compute difference from the model zeta RMS profile
            delta_list[n]=np.sum((RMS_sqg_interest-RMS_model_interest)**2)
      
        #Refine N_0^2 search:
        N02_min=N2_list[np.argmin(delta_list)]
        power = find_scientific_notation(N02_min)
        if N02_min-10**-power==0:
            N2_list_refine=np.concatenate((np.arange(9.1*10**-(power+1), 10**-power,10**-(power+2)),np.arange(N02_min+10**-(power+1),N02_min+10**-power, 10**-(power+1))))
        else:
            N2_list_refine=np.arange(N02_min-10**-power,N02_min+10**-power, 10**-(power+1))[1:-1]
            
        delta_list=np.zeros(len(N2_list_refine))+np.nan
        
        #Loop over the different N_0^2 values to try (in N2_list_refine)
        for N02, n in zip(N2_list_refine, np.arange(len(N2_list_refine))):
            N0=np.sqrt(N02)            
            N0of0=N0/f0
            Rho0oN02og=rho0/N02/g
            goN02orho0=g/N02/rho0
            rho_cst=N0of0*rho0
            zz= z_model*N0of0
        
            zeta = sqg_rel_vort(ssh_sqg, M, L, N, kk, zz, gof0)[:,50:-50, 50:-50] #Reduce the field to remove spectral residuals
            
            #Compute RMS
            RMS_one_day=np.sqrt(np.nanmean((zeta**2),axis=(1,2)))
            RMS_sqg_interest=RMS_one_day[np.where(mask_MLD)[0]]
            
            #Compute difference from the model zeta RMS profile
            delta_list[n]=np.sqrt(np.nanmean((RMS_sqg_interest-RMS_model_interest)**2))

        N2_min=N2_list_refine[np.argmin(delta_list)]
        N2_list_month[idd]=N2_min
        N02=N2_min
        print(N02)
        N0=np.sqrt(N02)        
        N0of0=N0/f0
        zz= z_model*N0of0
        
        zeta = sqg_rel_vort(ssh_sqg, M, L, N, kk, zz, gof0)[:,50:-50, 50:-50]
        RMS_sqg=np.sqrt(np.nanmean((zeta**2),axis=(1,2)))/f0
        
        RMS_model = (zeta_model**2).mean(dim=["x", "y"])**(1/2)/f0

        #Plot vorticity RMS profile from the model and the eSQG reconstruction
        fig=plt.figure()
        xmax = np.max(np.concatenate((RMS_sqg,RMS_model)))+.05
        plt.plot(RMS_model, z_model, c='k', label='Model')
        power = find_scientific_notation(N02)
        plt.plot(RMS_sqg, z_model, 'r', label='SQG ' + ' - N$^2$ = '+ str(np.round(N02*10**(power+2))/100 * 10**(-power)))
        plt.fill([0,2, 2, 0, 0], [-MLD_mean_month, -MLD_mean_month, 0, 0, -MLD_mean_month], 'b', alpha=.2)

        plt.legend(ncol=2)
        plt.ylim(-1000,0)
        plt.xlim(0,xmax)
        plt.grid()
        plt.xlabel('RMS $\zeta/f0$')
        plt.ylabel('depth (m)')
        fig.savefig(path_save_fig + month_label[m-1]+'_'+str(day_sel)+'_rms_zeta.png',bbox_inches='tight',dpi=300)
        plt.close()
        
        #Check reconstruction on physical maps - Plot maps of zeta from the model and the eSQG reconstruction at three different depths
        vscale=np.round(np.max([-np.min(zeta_model.isel(z=39).values/f0),np.max(zeta_model.isel(z=39).values/f0)])*100)/100 +.05
        vmin=-vscale
        vmax=vscale
        fig, axs = plt.subplots(2, 3, figsize=(8, 5))
        fig.suptitle('N$^2$ =' + str(np.round(N02*10**(power+2))/100 * 10**(-power)) + ' s$^{-2}$')
        axs=axs.flatten()
        
        axs[0].pcolormesh(lon2,lat2,zeta_model.isel(z=39)/f0, vmin=vmin,vmax=vmax, cmap='bwr')
        axs[0].set_title('$\zeta_{model}/f0$ at 253 m')
        axs[3].pcolormesh(lon2,lat2,-zeta[39,:,:]/f0, vmin=vmin,vmax=vmax, cmap='bwr')
        axs[3].set_title('$\zeta_{SQG}/f0$ at 253 m')
        axs[1].pcolormesh(lon2,lat2,zeta_model.isel(z=53)/f0, vmin=vmin,vmax=vmax, cmap='bwr')
        axs[1].set_title('$\zeta_{model}/f0$ at 592 m')
        axs[4].pcolormesh(lon2,lat2,-zeta[53,:,:]/f0, vmin=vmin,vmax=vmax, cmap='bwr')
        axs[4].set_title('$\zeta_{SQG}/f0$ at 592 m')
        axs[2].pcolormesh(lon2,lat2,zeta_model.isel(z=60)/f0, vmin=vmin,vmax=vmax, cmap='bwr')
        axs[2].set_title('$\zeta_{model}/f0$ at 899 m')
        im=axs[5].pcolormesh(lon2,lat2,-zeta[60,:,:]/f0, vmin=vmin,vmax=vmax, cmap='bwr')
        axs[5].set_title('$\zeta_{SQG}/f0$ at 899 m')
    
        axs[0].set_ylabel('Latitude')
        axs[3].set_ylabel('Latitude')
    
        for i in range(3):
            axs[i].set_xticklabels([])
            axs[i+3].set_xlabel('Longitude')
        
        for i in np.array([1,2,4,5]):
            axs[i].set_yticklabels([])
        fig.tight_layout()
        pos=axs[5].get_position()
        cax=fig.add_axes([pos.x0+pos.width+.02,pos.y0,0.02,2.76*pos.width])
        cb=plt.colorbar(im, cax=cax)
        cb.set_label('$\zeta/f0$')
        
        fig.savefig(path_save_fig + month_label[m-1]+'_'+str(day_sel)+'_maps_zeta.png',bbox_inches='tight',dpi=300)
        plt.close()

        #Compute spatial and spectral corelation at each depth
        spatial_corr = np.zeros(N)
        for dep in range(len(z_model)):
            A = -zeta[dep,:,:]/f0 #the eSQG reconstruction
            B = zeta_model.isel(z=dep).values/f0 #the model
            if len(np.where(np.isnan(B.reshape(-1,1))==1)[0])>0:
                x=np.arange(len(B[0,:]))
                y=np.arange(len(B[0,:]))
                x, y=np.meshgrid(x, y)
                x=np.squeeze(x.reshape(-1,1))
                y=np.squeeze(y.reshape(-1,1))
                idnotnan=np.where(np.isnan(B.reshape(-1,1))==0)[0]
                data_interp = griddata((x[idnotnan], y[idnotnan]), np.squeeze(B.reshape(-1,1))[idnotnan], (x, y), method='cubic').reshape(B.shape)
                B=data_interp
            spatial_corr[dep] = stats.pearsonr(A.ravel(), B.ravel())[0]

            #Detrend signals before computing their spectra
            A = signal.detrend(A,axis=0,type='linear')
            A = signal.detrend(A,axis=1,type='linear')
            B = signal.detrend(B,axis=0,type='linear')
            B = signal.detrend(B,axis=1,type='linear')

            #Compute spectra
            EuA,EuB,cs,coh,k,l,d1,d2,Atofft,Btofft = spectra_w(A,B,dx,dx)
            
            #Isospectra
            kiso,EiA,dkiso = calc_ispec(k,l,EuA[:,:])
            kiso,EiB,dkiso = calc_ispec(k,l,EuB[:,:])
            kiso,Ei_cs,dkiso = calc_ispec(k,l,cs[:,:])
            if dep==0:
                coh_2D=np.zeros([len(z_model),len(Ei_cs)])
            coh_2D[dep,:]=np.abs(Ei_cs)**2 / EiA / EiB
    
        #Make big figure combining iso-spectra at two different depths, the spectral coherence, 
        #the RMS profile and the correlation profile
        
        fig=plt.figure(figsize=(8,12))
        ax0=fig.add_axes([.05, .5, .45, .25])
        
        #Plot iso-spectra at two different depths
        #250 m
        dep=39
        A = -zeta[dep,:,:]/f0
        A = signal.detrend(A,axis=0,type='linear')
        A = signal.detrend(A,axis=1,type='linear')
        B = zeta_model.isel(z=dep)/f0
        B = signal.detrend(B,axis=0,type='linear')
        B = signal.detrend(B,axis=1,type='linear')
        EuA,EuB,cs,coh,k,l,d1,d2,Atofft,Btofft = spectra_w(A,B,dx,dx)
        
        #Isospectra
        kiso,EiA,dkiso = calc_ispec(k,l,EuA[:,:])
        kiso,EiB,dkiso = calc_ispec(k,l,EuB[:,:])
        
        ax0.plot(kiso,EiA, 'r--', label='SQG - 253 m')
        ax0.plot(kiso,EiB, 'r', label='Model - 253 m')

        #600 m
        dep=53
        A = -zeta[dep,:,:]/f0
        A = signal.detrend(A,axis=0,type='linear')
        A = signal.detrend(A,axis=1,type='linear')
        B = zeta_model.isel(z=dep)/f0
        B = signal.detrend(B,axis=0,type='linear')
        B = signal.detrend(B,axis=1,type='linear')
        EuA,EuB,cs,coh,k,l,d1,d2,Atofft,Btofft = spectra_w(A,B,dx,dx)
        
        #Isospectra
        kiso,EiA,dkiso = calc_ispec(k,l,EuA[:,:])
        kiso,EiB,dkiso = calc_ispec(k,l,EuB[:,:])
        
        ax0.plot(kiso,EiA, 'k--', label='SQG - 592 m')
        ax0.plot(kiso,EiB, 'k', label='Model - 592 m')
        ax0.plot([1/40,1/40], [1e-6,1e3], 'b') #filtering scale
        
        ax0.set_xscale('log') 
        ax0.set_yscale('log') 
        xmin, xmax = kiso.min(), 2e-1
        ax0.set_xlim(xmin, xmax) 
        plt.grid(True, which="both")
        ax0.legend(ncol=2)
        ax0.set_ylabel('power spectrum (/ cpkm)')
        ax0.set_xticklabels([])
    
        #Add the spectral coherence
        pos0=ax0.get_position()
        ax1=fig.add_axes([pos0.x0, 0.215, pos0.width, pos0.height])
        im = ax1.contourf(kiso,zeta_model.z, coh_2D,cmap='plasma', vmin=0, vmax=1, levels=np.arange(0,1.02,0.05))
        
        ax1.plot([1/40,1/40], [0,1000], 'w--')
        ymin, ymax=zeta_model.z.isel(z=0),zeta_model.z.isel(z=-1)
        ax1.set_ylim(ymin, ymax) 
        ax1.set_xlim(xmin, xmax) 
        ax1.set_xscale('log')
        ax1.invert_yaxis()
        ax1.set_xlabel('wavenumber (cpkm)')
        ax1.set_ylabel('depth (m)')
        
        #Add colorbar
        pos1=ax1.get_position()
        cax=fig.add_axes([pos1.x0+pos1.width+.02, pos1.y0, 0.02, pos1.height])
        cbar = fig.colorbar(im,cax=cax, orientation='vertical')
        cbar.set_label('Coherence')
        
        #Add RMS profile from zeta
        xmax = np.max(np.concatenate((RMS_sqg,RMS_model)))+.05
        ax2=fig.add_axes([pos1.x0+pos1.width+.17, pos1.y0, pos1.width/2, pos1.height])
        ax2.plot(RMS_model, -z_model, c='k', label='Model')
        ax2.plot(RMS_sqg, -z_model, 'k--', label='SQG ')
        ax2.fill([0,100, 100, 0, 0], [MLD_mean_month, MLD_mean_month, 0, 0, MLD_mean_month], 'b', alpha=.2)
        ax2.set_title('N$^2$ = '+str(np.round(N02*10**(power+1))/10**(power+1)))
        ax2.set_xlabel('RMS $\zeta$/f0')
        ax2.set_ylim(ymin, ymax) 
        ax2.set_xlim(0, xmax) 
        ax2.invert_yaxis()
        plt.grid()
    
        #Add spatial correlation
        pos2=ax2.get_position()
        ax3=fig.add_axes([pos2.x0, pos0.y0, pos2.width, pos2.height])
        ax3.plot(spatial_corr, -z_model, c='k')
        ax3.fill([0,100, 100, 0, 0], [MLD_mean_month, MLD_mean_month, 0, 0, MLD_mean_month], 'b', alpha=.2)
        ax3.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
        ax3.set_xlabel('Spatial correlation')
        ax3.xaxis.set_label_position('top') 
        ax3.set_ylim(ymin, ymax) 
        ax3.set_xlim(0, 1) 
        ax3.invert_yaxis()
        plt.grid()
        
        fig.savefig(path_save_fig + month_label[m-1]+'_'+str(day_sel)+'_spectra_zeta.png',bbox_inches='tight',dpi=300)
        plt.close()

        #Tune c on w
        w_model=w_filtered

        #Select w values associated to the tail of the distribution (+/-1 sigma, considering a normal distribution)
        abs_velocity_field = np.abs(w_model)
        thresholds = abs_velocity_field.reduce(np.nanpercentile, dim=['x', 'y'], q=68)
        
        #Create a mask for values above the threshold
        mask = abs_velocity_field >= thresholds
        masked_velocity_field_model = w_model.where(mask)

        #Compute RMS
        RMS_model_w=(masked_velocity_field_model**2).mean(dim=["x", "y"])**(1/2)
        RMS_model_interest=RMS_model_w.where(mask_MLD, drop=True)

        #Derive w from eSQG using c=1
        N0=np.sqrt(N02)
        cc=1
        N0of0=N0/f0
        gof0=g/f0
        Rho0oN02og=rho0/N02/g
        goN02orho0=g/N02/rho0
        rho_cst=N0of0*rho0
        zz= z_model*N0of0
    
        M=len(ssh_sqg[:,0])
        L=len(ssh_sqg[0,:])
        
        wref = sqg_w(ssh_sqg, cc, rho_cst, M, L, N, kk, kkx, kky, zz, goN02orho0, gof0)[:,50:-50, 50:-50]

        #Create a mask for values above the threshold
        masked_velocity_field_sqg = np.where(mask, wref, np.nan)
        
        c_list=np.arange(0.5,5,0.05)
        RMS_model=(masked_velocity_field_model**2).mean(dim=["x", "y"])**(1/2)
        RMS_model_interest=RMS_model.where(mask_MLD, drop=True).compute()
        rmse=np.zeros(len(c_list))
        
        #Loop over different c values (in c_list)
        for c,ic in zip(c_list,np.arange(len(c_list))):
            w = c*masked_velocity_field_sqg

            #Compute RMS profiles
            RMS_sqg=np.nanmean(w**2,axis=(1,2))**(1/2)
            RMS_sqg_interest=RMS_sqg[np.where(mask_MLD)[0]]

            #Compute difference from the model w RMS profile
            rmse[ic]=np.mean((RMS_sqg_interest-RMS_model_interest)**2)  
        cc_min=c_list[np.argmin(rmse)]
        
        cc_list_month[idd]=cc_min
        cc=cc_min
        w_sqg = sqg_w(ssh_sqg, cc, rho_cst, M, L, N, kk, kkx, kky, zz, goN02orho0, gof0)[:,50:-50, 50:-50]
        masked_velocity_field_sqg = np.where(mask, w_sqg, np.nan)
        RMS_sqg=np.nanmean(masked_velocity_field_sqg**2,axis=(1,2))**(1/2)

        #Plot w RMS profile from the model and the eSQG reconstruction
        xmax = ds_select.RMS_model.max().values+10
        fig=plt.figure()
        for t in range(len(ds_select.time_counter)):
            plt.plot(ds_select.RMS_model.isel(time_counter=t), -ds_RMS_model.z, c='grey', linewidth=.1)
        plt.plot(RMS_model, z_model, c='k', label='Model')
        plt.plot(RMS_sqg, z_model, 'r', label='SQG ' + ' - N$^2$ =' + str(np.round(N02*10**(power+2))/100 * 10**(-power)) + ' s$^{-2}$ - ' + 'c = ' + str(np.round(cc*100)/100))
        plt.fill([0,100, 100, 0, 0], [-MLD_mean_month, -MLD_mean_month, 0, 0, -MLD_mean_month], 'b', alpha=.2)
        
        plt.legend()
        plt.ylim(-1000,0)
        plt.xlim(0,xmax)
        plt.grid()
        plt.xlabel('RMS w (m day$^{-1}$)')
        plt.ylabel('depth (m)')
        
        fig.savefig(path_save_fig + month_label[m-1]+'_'+str(day_sel)+'_rms_w.png',bbox_inches='tight',dpi=300)
        plt.close()

        #Check reconstruction on physical maps - Plot maps of zeta from the model and the eSQG reconstruction at three different depths
        vscale=np.round(np.max([-np.min(w_filtered.isel(z=39).values),np.max(w_filtered.isel(z=39).values)])*100)/100 +10
        vmin=-vscale
        vmax=vscale
        
        fig, axs = plt.subplots(2, 3, figsize=(8, 5))
        fig.suptitle('N$^2$ =' + str(np.round(N02*10**(power+2))/100 * 10**(-power)) + ' s$^{-2}$ - ' + 'c = ' + str(np.round(cc*100)/100))
        axs=axs.flatten()

        axs[0].pcolormesh(lon2,lat2,w_filtered.isel(z=39), vmin=vmin,vmax=vmax, cmap='bwr')
        axs[0].set_title('w$_{model}$ at 253 m')
        axs[3].pcolormesh(lon2,lat2,w_sqg[39,:,:], vmin=vmin,vmax=vmax, cmap='bwr')
        axs[3].set_title('w$_{SQG}$ at 253 m')
        axs[1].pcolormesh(lon2,lat2,w_filtered.isel(z=53), vmin=vmin,vmax=vmax, cmap='bwr')
        axs[1].set_title('w$_{model}$ at 592 m')
        axs[4].pcolormesh(lon2,lat2,w_sqg[53,:,:], vmin=vmin,vmax=vmax, cmap='bwr')
        axs[4].set_title('w$_{SQG}$ at 592 m')
        axs[2].pcolormesh(lon2,lat2,w_filtered.isel(z=60), vmin=vmin,vmax=vmax, cmap='bwr')
        axs[2].set_title('w$_{model}$ at 899 m')
        im=axs[5].pcolormesh(lon2,lat2,w_sqg[60,:,:], vmin=vmin,vmax=vmax, cmap='bwr')
        axs[5].set_title('w$_{SQG}$ at 899 m')
        
        axs[0].set_ylabel('Latitude')
        axs[3].set_ylabel('Latitude')
        
        for i in range(3):
            axs[i].set_xticklabels([])
            axs[i+3].set_xlabel('Longitude')
        
        for i in np.array([1,2,4,5]):
            axs[i].set_yticklabels([])
        fig.tight_layout()
        pos=axs[5].get_position()
        cax=fig.add_axes([pos.x0+pos.width+.02,pos.y0,0.02,2.76*pos.width])
        cb=plt.colorbar(im, cax=cax)
        cb.set_label('w (m day$^{-1}$)')
        
        fig.savefig(path_save_fig + month_label[m-1]+'_'+str(day_sel)+'_maps_w.png',bbox_inches='tight',dpi=300)
        plt.close()

        spatial_corr = np.zeros(N)
        for dep in range(len(z_model)):
            A = w_sqg[dep,:,:]
            B = w_filtered.isel(z=dep).values
            if len(np.where(np.isnan(B.reshape(-1,1))==1)[0])>0:
                x=np.arange(len(B[0,:]))
                y=np.arange(len(B[0,:]))
                x, y=np.meshgrid(x, y)
                x=np.squeeze(x.reshape(-1,1))
                y=np.squeeze(y.reshape(-1,1))
                idnotnan=np.where(np.isnan(B.reshape(-1,1))==0)[0]
                data_interp = griddata((x[idnotnan], y[idnotnan]), np.squeeze(B.reshape(-1,1))[idnotnan], (x, y), method='cubic').reshape(B.shape)
                B=data_interp
            spatial_corr[dep] = stats.pearsonr(A.ravel(), B.ravel())[0]
            A = signal.detrend(A,axis=0,type='linear')
            A = signal.detrend(A,axis=1,type='linear')
            B = signal.detrend(B,axis=0,type='linear')
            B = signal.detrend(B,axis=1,type='linear')
            EuA,EuB,cs,coh,k,l,d1,d2,Atofft,Btofft = spectra_w(A,B,dx,dx)
            #Isospectra
            kiso,EiA,dkiso = calc_ispec(k,l,EuA[:,:])
            kiso,EiB,dkiso = calc_ispec(k,l,EuB[:,:])
            kiso,Ei_cs,dkiso = calc_ispec(k,l,cs[:,:])
            if dep==0:
                coh_2D=np.zeros([len(z_model),len(Ei_cs)])
            coh_2D[dep,:]=np.abs(Ei_cs)**2 / EiA / EiB
    
        #Make big figure combining iso-spectra at two different depths, the spectral coherence, 
        #the RMS profile and the correlation profile
        
        fig=plt.figure(figsize=(8,12))
        ax0=fig.add_axes([.05, .5, .45, .25])
        
        #Plot iso-spectra at two different depths
        #250 m
        dep=39
        A = w_sqg[dep,:,:]
        A = signal.detrend(A,axis=0,type='linear')
        A = signal.detrend(A,axis=1,type='linear')
        B = w_filtered.isel(z=dep)
        B = signal.detrend(B,axis=0,type='linear')
        B = signal.detrend(B,axis=1,type='linear')
        EuA,EuB,cs,coh,k,l,d1,d2,Atofft,Btofft = spectra_w(A,B,dx,dx)
        #Isospectra
        kiso,EiA,dkiso = calc_ispec(k,l,EuA[:,:])
        kiso,EiB,dkiso = calc_ispec(k,l,EuB[:,:])
        
        ax0.plot(kiso,EiA, 'r--', label='SQG - 253 m')
        ax0.plot(kiso,EiB, 'r', label='Model - 253 m')
        
        #600 m
        dep=53
        A = w_sqg[dep,:,:]
        A = signal.detrend(A,axis=0,type='linear')
        A = signal.detrend(A,axis=1,type='linear')
        B = w_filtered.isel(z=dep)
        B = signal.detrend(B,axis=0,type='linear')
        B = signal.detrend(B,axis=1,type='linear')
        EuA,EuB,cs,coh,k,l,d1,d2,Atofft,Btofft = spectra_w(A,B,dx,dx)
        
        #Isospectra
        kiso,EiA,dkiso = calc_ispec(k,l,EuA[:,:])
        kiso,EiB,dkiso = calc_ispec(k,l,EuB[:,:])
        
        ax0.plot(kiso,EiA, 'k--', label='SQG - 592 m')
        ax0.plot(kiso,EiB, 'k', label='Model - 592 m')
        ax0.plot([1/40,1/40], [1e-1,1e6], 'b')
    
        ax0.set_xscale('log') 
        ax0.set_yscale('log') 
        xmin, xmax = kiso.min(), 2e-1
        ax0.set_xlim(xmin, xmax) 
        plt.grid(True, which="both")
        ax0.legend(ncol=2)
        ax0.set_ylabel('power spectrum ((m/s)$^2$/ cpkm)')
        ax0.set_xticklabels([])
        
        #Add the spectral coherence
        pos0=ax0.get_position()
        ax1=fig.add_axes([pos0.x0, 0.215, pos0.width, pos0.height])
        im = ax1.contourf(kiso,zeta_model.z, coh_2D,cmap='plasma', vmin=0, vmax=1, levels=np.arange(0,1.02,0.05))
        ax1.plot([1/40,1/40], [0,1000], 'w--')
        ymin, ymax=zeta_model.z.isel(z=0),zeta_model.z.isel(z=-1)
        ax1.set_ylim(ymin, ymax) 
        ax1.set_xlim(xmin, xmax) 
        ax1.set_xscale('log')
        ax1.invert_yaxis()
        ax1.set_xlabel('wavenumber (cpkm)')
        ax1.set_ylabel('depth (m)')
        
        #Add colorbar
        pos1=ax1.get_position()
        cax=fig.add_axes([pos1.x0+pos1.width+.02, pos1.y0, 0.02, pos1.height])
        cbar = fig.colorbar(im,cax=cax, orientation='vertical')
        cbar.set_label('Coherence')
    
        #Add RMS profile of w
        ax2=fig.add_axes([pos1.x0+pos1.width+.17, pos1.y0, pos1.width/2, pos1.height])
        #Add MLD day
        ax2.fill([0,100, 100, 0, 0], [MLD_mean_month, MLD_mean_month, 0, 0, MLD_mean_month], 'b', alpha=.2)
        xmax = ds_select.RMS_model.max().values+10
        for t in range(len(ds_select.time_counter)):
            ax2.plot(ds_select.RMS_model.isel(time_counter=t), ds_RMS_model.z, c='grey', linewidth=.1)
        ax2.plot(RMS_model, -z_model, c='k', label='Model')
        ax2.plot(RMS_sqg, -z_model, 'k--', label='SQG ')
        ratio = cc/N0
        ax2.set_title('N$^2$ = '+str(np.round(N02*10**(power+1))/10**(power+1)) + ', ' + 'c = ' + str(np.round(cc*100)/100) + ', c/N = ' + str(np.ceil(ratio)))
        #ax2.legend(ncol=2)
        ax2.set_xlabel('RMS w')
        ax2.set_ylim(ymin, ymax) 
        ax2.set_xlim(0, xmax) 
        ax2.invert_yaxis()
        plt.grid()
        
        #Add spatial correlation
        pos2=ax2.get_position()
        ax3=fig.add_axes([pos2.x0, pos0.y0, pos2.width, pos2.height])
        ax3.plot(spatial_corr, -z_model, c='k')
        ax3.fill([0,100, 100, 0, 0], [MLD_mean_month, MLD_mean_month, 0, 0, MLD_mean_month], 'b', alpha=.2)
        ax3.tick_params(top=True, labeltop=True, bottom=False, labelbottom=False)
        ax3.set_xlabel('Spatial correlation')
        ax3.xaxis.set_label_position('top') 
        ax3.set_ylim(ymin, ymax) 
        ax3.set_xlim(0, 1) 
        ax3.invert_yaxis()
        plt.grid()
    
        fig.savefig(path_save_fig + month_label[m-1]+'_'+str(day_sel)+'_spectra_w.png',bbox_inches='tight',dpi=300)
        plt.close()

    #Save the values of N_0^2 and c
    ds_to_save = xr.Dataset(
        data_vars=dict(
            N2=(["day"], N2_list_month),
            cc=(["day"], cc_list_month),
            day_number=(["day"], np.argsort(RMSE_anomaly.values)[:10])
        ),
        coords=dict(
            day=("day", np.arange(10),
        ) 
    ))

    ds_to_save.to_netcdf(path=path_save_fig + month_label[m-1]+'.nc')

/gxfs_home/geomar/smomw691/miniconda3/envs/py3_std/lib/python3.7/site-packages/xarray/core/indexing.py:1227: PerformanceWarning: Slicing is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array[indexer]

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array[indexer]
  return self.array[key]


20
